# Fine-tune PhoBERT for NER (PII Detection)

- **Model**: `vinai/phobert-base` with VnCoreNLP word segmentation
- **Data**: `quynong/cs419-data` from HuggingFace
- **Evaluation**:
  1. **Classification**: Binary (has entity vs no entity)
  2. **NER**: Micro Precision / Recall / F1 at entity level

In [1]:
!pip install -q transformers datasets seqeval accelerate vncorenlp py_vncorenlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 34.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.7 MB/s eta 0:00:00


In [2]:
import json
import numpy as np
import torch
from pathlib import Path
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## 1. Load Dataset from HuggingFace

In [3]:
dataset = load_dataset("quynong/cs419-data")
print(dataset)
print(f"\nTrain sample:")
print(dataset['train'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/620 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.76M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.09M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/54117 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6014 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source_text', 'language', 'privacy_mask'],
        num_rows: 54117
    })
    validation: Dataset({
        features: ['source_text', 'language', 'privacy_mask'],
        num_rows: 6014
    })
})

Train sample:
{'source_text': 'Chúng tôi có những lo ngại về một số giao dịch từ tài khoản 0041000990011 (Vietcombank), IBAN PK51MGLA0900120022020017. Vui lòng kiểm tra nhật ký từ 08/05/2022 đến Ngày 2/2/1951.', 'language': 'vi', 'privacy_mask': [{'start': 60, 'end': 87, 'label': 'ACCOUNTNUMBER', 'value': '0041000990011 (Vietcombank)'}, {'start': 94, 'end': 118, 'label': 'IBAN', 'value': 'PK51MGLA0900120022020017'}, {'start': 149, 'end': 159, 'label': 'DATE', 'value': '08/05/2022'}, {'start': 164, 'end': 177, 'label': 'DATE', 'value': 'Ngày 2/2/1951'}]}


## 2. Setup PhoBERT Tokenizer & Word Segmentation

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import py_vncorenlp

# Download VnCoreNLP models (only need to run once)
py_vncorenlp.download_model(save_dir='/content/drive/MyDrive/CS419')
segmenter = py_vncorenlp.VnCoreNLP(save_dir='/content/drive/MyDrive/CS419')

MODEL_NAME = "vinai/phobert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer: vinai/phobert-base
Vocab size: 64000


## 3. Build Label Set from Data

In [7]:
# Collect all unique entity labels
all_labels = set()
for split in dataset:
    for sample in dataset[split]:
        for ent in sample['privacy_mask']:
            all_labels.add(ent['label'])

all_labels = sorted(all_labels)
print(f"Found {len(all_labels)} entity types:")
print(all_labels)

# Build BIO label list
label_list = ["O"]
for lbl in all_labels:
    label_list.append(f"B-{lbl}")
    label_list.append(f"I-{lbl}")

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}
num_labels = len(label_list)

print(f"\nTotal BIO labels: {num_labels}")
print(f"First 10: {label_list[:10]}")

Found 54 entity types:
['ACCOUNTNAME', 'ACCOUNTNUMBER', 'AGE', 'AMOUNT', 'BIC', 'BITCOINADDRESS', 'BUILDINGNUMBER', 'CCCD', 'CITY', 'COMPANYNAME', 'COUNTY', 'CREDITCARDCVV', 'CREDITCARDISSUER', 'CREDITCARDNUMBER', 'CURRENCY', 'CURRENCYCODE', 'CURRENCYNAME', 'CURRENCYSYMBOL', 'DATE', 'DOB', 'EMAIL', 'ETHEREUMADDRESS', 'EYECOLOR', 'FIRSTNAME', 'GENDER', 'HEIGHT', 'IBAN', 'IPADDRESS', 'JOBAREA', 'JOBTITLE', 'JOBTYPE', 'LASTNAME', 'LITECOINADDRESS', 'MAC', 'MASKEDNUMBER', 'MIDDLENAME', 'NEARBYGPSCOORDINATE', 'ORDINALDIRECTION', 'PASSWORD', 'PHONEIMEI', 'PHONENUMBER', 'PIN', 'PREFIX', 'SECONDARYADDRESS', 'SEX', 'STATE', 'STREET', 'TIME', 'URL', 'USERAGENT', 'USERNAME', 'VEHICLEVIN', 'VEHICLEVRM', 'ZIPCODE']

Total BIO labels: 109
First 10: ['O', 'B-ACCOUNTNAME', 'I-ACCOUNTNAME', 'B-ACCOUNTNUMBER', 'I-ACCOUNTNUMBER', 'B-AGE', 'I-AGE', 'B-AMOUNT', 'I-AMOUNT', 'B-BIC']


## 4. Tokenize & Align Labels

For each sample:
1. Segment Vietnamese text with VnCoreNLP
2. Tokenize with PhoBERT tokenizer
3. Align character-level entity spans to subword tokens using BIO scheme

In [25]:
MAX_LENGTH = 256

def segment_text(text):
    """Word-segment Vietnamese text using VnCoreNLP."""
    try:
        sentences = segmenter.word_segment(text)
        return " ".join(sentences)
    except:
        return text


def char_to_token_labels(source_text, privacy_mask, encoding, segmented_text):
    """
    Convert character-level entity spans to token-level BIO labels.
    Uses offset_mapping from tokenizer to align.
    """
    # Build character-level label array for the ORIGINAL text
    char_labels = ['O'] * len(source_text)
    for ent in privacy_mask:
        start, end, label = ent['start'], ent['end'], ent['label']
        if start >= len(source_text) or end > len(source_text):
            continue
        char_labels[start] = f"B-{label}"
        for i in range(start + 1, end):
            char_labels[i] = f"I-{label}"

    # Map from segmented text positions back to original text positions
    # VnCoreNLP replaces spaces within words with '_' and keeps structure
    # We need a char map: segmented_pos -> original_pos
    seg_to_orig = []
    orig_idx = 0
    for seg_idx, seg_char in enumerate(segmented_text):
        if seg_char == '_' and orig_idx < len(source_text) and source_text[orig_idx] == ' ':
            seg_to_orig.append(orig_idx)
            orig_idx += 1
        elif seg_char == ' ':
            # Extra space added by segmenter between words
            # Check if original also has space
            if orig_idx < len(source_text) and source_text[orig_idx] == ' ':
                seg_to_orig.append(orig_idx)
                orig_idx += 1
            else:
                seg_to_orig.append(-1)  # no mapping
        else:
            if orig_idx < len(source_text):
                seg_to_orig.append(orig_idx)
                orig_idx += 1
            else:
                seg_to_orig.append(-1)

    # Now assign labels to each token using offset_mapping
    offset_mapping = encoding.get('offset_mapping', [])
    token_labels = []

    prev_label = 'O'
    for (tok_start, tok_end) in offset_mapping:
        if tok_start == 0 and tok_end == 0:
            # Special token
            token_labels.append(-100)
            continue

        # Get the original char positions for this token span
        orig_positions = []
        for seg_pos in range(tok_start, min(tok_end, len(seg_to_orig))):
            if seg_pos < len(seg_to_orig) and seg_to_orig[seg_pos] >= 0:
                orig_positions.append(seg_to_orig[seg_pos])

        if not orig_positions:
            token_labels.append(label2id['O'])
            prev_label = 'O'
            continue

        # Use the label of the first character of this token
        first_orig_pos = orig_positions[0]
        if first_orig_pos < len(char_labels):
            lbl = char_labels[first_orig_pos]
        else:
            lbl = 'O'

        if lbl in label2id:
            token_labels.append(label2id[lbl])
        else:
            token_labels.append(label2id['O'])
        prev_label = lbl

    return token_labels

print("Label alignment function loaded.")

Label alignment function loaded.


In [26]:
def tokenize_and_align(examples):
    """
    Tokenize batch of examples and align entity labels to subword tokens.
    """
    all_input_ids = []
    all_attention_mask = []
    all_labels = []

    for source_text, privacy_mask in zip(examples['source_text'], examples['privacy_mask']):
        # Step 1: Word segment
        segmented = segment_text(source_text)

        # Step 2: Tokenize (Bỏ return_offsets_mapping vì PhoBERT không hỗ trợ)
        encoding = tokenizer(
            segmented,
            max_length=MAX_LENGTH,
            truncation=True,
            padding='max_length',
        )

        # --- BƯỚC BỔ SUNG: TỰ TẠO OFFSET MAPPING CHO PHOBERT ---
        input_ids = encoding['input_ids']
        tokens = tokenizer.convert_ids_to_tokens(input_ids)

        offset_mapping = []
        curr_pos = 0

        for token in tokens:
            # 1. Các token đặc biệt (<s>, </s>, <pad>, <unk>) cho offset (0, 0)
            if token in tokenizer.all_special_tokens:
                offset_mapping.append((0, 0))
                continue

            # 2. Xóa bỏ ký tự '@@' đặc trưng của PhoBERT subword để lấy độ dài thực
            clean_token = token.replace("@@", "")

            # 3. Bỏ qua các khoảng trắng dư thừa trong chuỗi segmented
            while curr_pos < len(segmented) and segmented[curr_pos] == ' ':
                curr_pos += 1

            start = curr_pos
            end = curr_pos + len(clean_token)

            offset_mapping.append((start, end))
            curr_pos = end

        # Nhúng offset_mapping tự tạo vào encoding để hàm char_to_token_labels sử dụng
        encoding['offset_mapping'] = offset_mapping
        # --------------------------------------------------------

        # Step 3: Align labels (Giữ nguyên hàm của bạn, nó đã hoạt động rất tốt!)
        token_labels = char_to_token_labels(
            source_text, privacy_mask, encoding, segmented
        )

        # Pad labels to MAX_LENGTH
        while len(token_labels) < MAX_LENGTH:
            token_labels.append(-100)
        token_labels = token_labels[:MAX_LENGTH]

        all_input_ids.append(encoding['input_ids'])
        all_attention_mask.append(encoding['attention_mask'])
        all_labels.append(token_labels)

    return {
        'input_ids': all_input_ids,
        'attention_mask': all_attention_mask,
        'labels': all_labels,
    }

print("Tokenization function loaded & patched for PhoBERT.")

Tokenization function loaded & patched for PhoBERT.


In [27]:
# Apply tokenization to all splits
tokenized_dataset = dataset.map(
    tokenize_and_align,
    batched=True,
    batch_size=32,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing",
)

print(tokenized_dataset)
print(f"\nSample input_ids length: {len(tokenized_dataset['train'][0]['input_ids'])}")
print(f"Sample labels length: {len(tokenized_dataset['train'][0]['labels'])}")

Tokenizing:   0%|          | 0/54117 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/6014 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 54117
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6014
    })
})

Sample input_ids length: 256
Sample labels length: 256


## 5. Define Model & Training

In [28]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
model.to(device)
print(f"Model loaded: {MODEL_NAME} with {num_labels} labels")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: vinai/phobert-base with 109 labels
Parameters: 134,491,501


In [29]:
training_args = TrainingArguments(
    output_dir="./phobert-ner-pii",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print("Training arguments configured.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training arguments configured.


## 6. Evaluation Metrics

Two evaluation modes:
1. **NER (Entity-level)**: seqeval micro P/R/F1
2. **Classification (Sentence-level)**: Does the sentence contain any entity?

In [30]:
def compute_metrics(eval_preds):
    """
    Compute both NER entity-level and sentence-level classification metrics.
    """
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # ──── NER Entity-level metrics (seqeval) ────────────────────────────────
    true_labels = []
    true_predictions = []

    for pred_seq, label_seq in zip(predictions, labels):
        pred_tags = []
        true_tags = []
        for p, l in zip(pred_seq, label_seq):
            if l == -100:
                continue
            pred_tags.append(id2label[p])
            true_tags.append(id2label[l])
        true_predictions.append(pred_tags)
        true_labels.append(true_tags)

    ner_precision = precision_score(true_labels, true_predictions, average='micro')
    ner_recall = recall_score(true_labels, true_predictions, average='micro')
    ner_f1 = f1_score(true_labels, true_predictions, average='micro')

    # ──── Classification: sentence has entity or not ────────────────────────
    cls_true = []
    cls_pred = []

    for pred_seq, label_seq in zip(predictions, labels):
        # True: does this sentence have any non-O label?
        has_entity_true = any(
            l not in (-100, label2id['O']) for l in label_seq
        )
        # Pred: does prediction contain any non-O?
        has_entity_pred = any(
            p != label2id['O'] for p, l in zip(pred_seq, label_seq) if l != -100
        )
        cls_true.append(int(has_entity_true))
        cls_pred.append(int(has_entity_pred))

    cls_precision, cls_recall, cls_f1, _ = precision_recall_fscore_support(
        cls_true, cls_pred, average='binary', zero_division=0
    )
    cls_accuracy = accuracy_score(cls_true, cls_pred)

    return {
        # NER entity-level
        "precision": ner_precision,
        "recall": ner_recall,
        "f1": ner_f1,
        # Classification sentence-level
        "cls_accuracy": cls_accuracy,
        "cls_precision": cls_precision,
        "cls_recall": cls_recall,
        "cls_f1": cls_f1,
    }

print("Metrics function loaded.")

Metrics function loaded.


In [31]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

print("Trainer ready. Starting training...")
trainer.train()

Trainer ready. Starting training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Cls Accuracy,Cls Precision,Cls Recall,Cls F1
1,0.061026,0.045560,0.917633,0.931617,0.924572,0.999834,0.999702,1.000000,0.999851
2,0.048551,0.033533,0.934113,0.947627,0.940822,0.999834,0.999702,1.000000,0.999851
3,0.039800,0.029464,0.942130,0.953125,0.947596,1.000000,1.000000,1.000000,1.000000
4,0.030504,0.025570,0.953831,0.960455,0.957132,1.000000,1.000000,1.000000,1.000000
5,0.007080,0.026856,0.955132,0.962963,0.959032,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=16915, training_loss=0.11248190156320408, metrics={'train_runtime': 4204.914, 'train_samples_per_second': 64.35, 'train_steps_per_second': 4.023, 'total_flos': 3.538568980267776e+16, 'train_loss': 0.11248190156320408, 'epoch': 5.0})

## 7. Final Evaluation

In [32]:
# Run final evaluation on validation set
eval_results = trainer.evaluate()

print("="*60)
print("FINAL EVALUATION RESULTS")
print("="*60)
print(f"\n{'─'*40}")
print("NER Entity-Level (Micro):")
print(f"  Precision: {eval_results['eval_precision']:.4f}")
print(f"  Recall:    {eval_results['eval_recall']:.4f}")
print(f"  F1:        {eval_results['eval_f1']:.4f}")
print(f"\n{'─'*40}")
print("Classification (Has Entity):")
print(f"  Accuracy:  {eval_results['eval_cls_accuracy']:.4f}")
print(f"  Precision: {eval_results['eval_cls_precision']:.4f}")
print(f"  Recall:    {eval_results['eval_cls_recall']:.4f}")
print(f"  F1:        {eval_results['eval_cls_f1']:.4f}")
print("="*60)

FINAL EVALUATION RESULTS

────────────────────────────────────────
NER Entity-Level (Micro):
  Precision: 0.9551
  Recall:    0.9630
  F1:        0.9590

────────────────────────────────────────
Classification (Has Entity):
  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1:        1.0000


In [33]:
# Detailed NER report per entity type
eval_output = trainer.predict(tokenized_dataset['validation'])
predictions = np.argmax(eval_output.predictions, axis=-1)
labels = eval_output.label_ids

true_labels = []
true_predictions = []

for pred_seq, label_seq in zip(predictions, labels):
    pred_tags = []
    true_tags = []
    for p, l in zip(pred_seq, label_seq):
        if l == -100:
            continue
        pred_tags.append(id2label[p])
        true_tags.append(id2label[l])
    true_predictions.append(pred_tags)
    true_labels.append(true_tags)

report = classification_report(true_labels, true_predictions, digits=4)
print("Detailed NER Classification Report (per entity type):")
print(report)

Detailed NER Classification Report (per entity type):
                     precision    recall  f1-score   support

        ACCOUNTNAME     0.9789    0.9915    0.9851       234
      ACCOUNTNUMBER     0.9921    0.9921    0.9921       253
                AGE     0.9708    0.9540    0.9623       174
             AMOUNT     0.9873    0.9831    0.9852       237
                BIC     0.9828    1.0000    0.9913        57
     BITCOINADDRESS     0.9779    0.9888    0.9833       179
     BUILDINGNUMBER     0.9910    0.9735    0.9821       226
               CCCD     0.9927    0.9784    0.9855       139
               CITY     0.9853    0.9853    0.9853       204
        COMPANYNAME     0.9340    0.9583    0.9460       192
             COUNTY     0.9910    1.0000    0.9955       221
      CREDITCARDCVV     0.9524    1.0000    0.9756        60
   CREDITCARDISSUER     0.9633    0.9813    0.9722       107
   CREDITCARDNUMBER     0.8495    0.9576    0.9003       165
           CURRENCY     0.9679

## 8. Save Model

In [34]:
# Save the best model
save_path = "/content/drive/MyDrive/CS419/phobert-ner-pii/best_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to: {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/drive/MyDrive/CS419/phobert-ner-pii/best_model


## 9. Inference Example

In [36]:
def predict_entities(text):
    """Run NER inference on a single Vietnamese text."""
    # Segment
    segmented = segment_text(text)

    # Tokenize (Bỏ return_offsets_mapping vì PhoBERT không hỗ trợ)
    encoding = tokenizer(
        segmented,
        max_length=MAX_LENGTH,
        truncation=True,
        return_tensors='pt'
    )

    # --- BƯỚC BỔ SUNG: TỰ TẠO OFFSET MAPPING ---
    input_ids = encoding['input_ids'][0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    offset_mapping = []
    curr_pos = 0

    for token in tokens:
        if token in tokenizer.all_special_tokens:
            offset_mapping.append((0, 0))
            continue

        clean_token = token.replace("@@", "")

        while curr_pos < len(segmented) and segmented[curr_pos] == ' ':
            curr_pos += 1

        start = curr_pos
        end = curr_pos + len(clean_token)

        offset_mapping.append((start, end))
        curr_pos = end
    # --------------------------------------------

    # Chuẩn bị input cho model (không cần pop offset_mapping nữa)
    inputs = {k: v.to(device) for k, v in encoding.items()}

    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    preds = torch.argmax(outputs.logits, dim=-1)[0].cpu().tolist()

    # Extract entities from BIO predictions
    entities = []
    current_entity = None

    for idx, (pred_id, (start, end)) in enumerate(zip(preds, offset_mapping)):
        if start == 0 and end == 0:
            continue
        label = id2label[pred_id]

        if label.startswith('B-'):
            if current_entity:
                entities.append(current_entity)
            current_entity = {
                'label': label[2:],
                'start': start,
                'end': end,
                'text': segmented[start:end],
            }
        elif label.startswith('I-') and current_entity:
            current_entity['end'] = end
            current_entity['text'] = segmented[current_entity['start']:end]
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None

    if current_entity:
        entities.append(current_entity)

    # Làm sạch text (xóa dấu '_' của VnCoreNLP để in ra giống văn bản gốc)
    for ent in entities:
        ent['text'] = ent['text'].replace('_', ' ')

    return entities


# Test
test_text = "Xin chào, tôi là Nguyễn Văn An, số điện thoại 0912345678, địa chỉ 123 Lê Lợi, Quận 1."
entities = predict_entities(test_text)
print(f"Text: {test_text}")
print(f"\nEntities found: {len(entities)}")
for ent in entities:
    print(f"  [{ent['label']}] \"{ent['text']}\" (pos {ent['start']}-{ent['end']})")

Text: Xin chào, tôi là Nguyễn Văn An, số điện thoại 0912345678, địa chỉ 123 Lê Lợi, Quận 1.

Entities found: 4
  [LASTNAME] "Nguyễn Văn An" (pos 18-31)
  [PHONENUMBER] "0912345678" (pos 48-58)
  [BUILDINGNUMBER] "123 Lê Lợi" (pos 69-79)
  [COUNTY] "Quận 1" (pos 82-88)
